In [1]:
from web3 import Web3
import json

In [2]:
rpcServer = 'HTTP://127.0.0.1:7545'
w3 = Web3(Web3.HTTPProvider(rpcServer))

contributionSC = open('./Solidity/build/contracts/Reward.json')
contributionData = json.load(contributionSC)
contributionAbi = contributionData['abi']
addressContribution = contributionData['networks']['5777']['address']
contribution_contract_instance = w3.eth.contract(address=addressContribution, abi=contributionAbi)

In [ ]:
#add all account balance 100
for i in range(1, 9):
    account = w3.eth.accounts[i]
    contribution_contract_instance.functions.addBalance(account,100).transact({'from': w3.eth.accounts[0]})

In [3]:
#check all account balance
for i in range(1, 9):
    account = w3.eth.accounts[i]
    balance = contribution_contract_instance.functions.balanceOf(account).call()
    print(account, balance)
    

0xae0a5a3Dbf8AB60f7B253514495f95C4574876d7 5507
0xE295e5D17dbae5FD80b4aBF64678aeA216a9D06d 2367
0x67e408fCD53738c39a94220e911204f3c90A787D 2363
0x4685B1D9c63717d7060ad689AF77Ba98dFc895E9 2308
0x36D38F3533690B71e1Cc60C9dAbd3A3543813596 2518
0xE2200280bd7bDc3bB44cA1Db822bBBF5766d358E 120
0x30B5191b7AaF03BED8c50Ebb960B64529e688a74 100
0x75caF2D330F75ea4b5d0c76067a218E2f0b82992 130


In [4]:
contribution_contract_instance.functions.addBalance(w3.eth.accounts[1],1000).transact({'from': w3.eth.accounts[0]})

HexBytes('0xd1a43db0e9ee5e137becf2c0ef82bdd028c7543c0316dd0b2d651ba38e822471')

In [16]:
name=contribution_contract_instance.functions.balanceOf(w3.eth.accounts[1]).call()
print(name)

2100


In [4]:
acc=w3.eth.account.from_key('0xd5ee1a861622cbbc27f6c2bb446c51067015692e03e0cbecddc07584a7676af6')
print(acc.address)
name=contribution_contract_instance.functions.balanceOf(acc.address).call()
print(name)

0xae0a5a3Dbf8AB60f7B253514495f95C4574876d7
2100


In [6]:
name=contribution_contract_instance.functions.balanceOf("0xae0a5a3Dbf8AB60f7B253514495f95C4574876d7").call()
print(name)

2100


In [4]:
name=contribution_contract_instance.functions.balanceOf("0x75caF2D330F75ea4b5d0c76067a218E2f0b82992").call()
print(name)

100


In [5]:
print(w3.is_address("0xA09aF9Ef981aD1C6dEDA7D4C4f03bE548cD960ea"))

True


In [5]:
from UitFedSecAggre.vanilla_system.Solidity.reward_service import RewardService
reward_service = RewardService()

In [6]:
#reward_service.pay("0x75caF2D330F75ea4b5d0c76067a218E2f0b82992", 30)
reward_service.getBalance('0x36D38F3533690B71e1Cc60C9dAbd3A3543813596')

130

In [8]:
#Kiểm tra số dư ban đầu của các client
for i in range(1, 6):
    account = w3.eth.accounts[i]
    balance = contribution_contract_instance.functions.balanceOf(account).call()
    print(account, f"client{i}", balance)

0xae0a5a3Dbf8AB60f7B253514495f95C4574876d7 client1 6724
0xE295e5D17dbae5FD80b4aBF64678aeA216a9D06d client2 4750
0x67e408fCD53738c39a94220e911204f3c90A787D client3 8612
0x4685B1D9c63717d7060ad689AF77Ba98dFc895E9 client4 4825
0x36D38F3533690B71e1Cc60C9dAbd3A3543813596 client5 7229


In [7]:
#Kiểm tra số dư lúc sau của các client
for i in range(1, 6):
    account = w3.eth.accounts[i]
    balance = contribution_contract_instance.functions.balanceOf(account).call()
    print(account, f"client{i}", balance)

0xae0a5a3Dbf8AB60f7B253514495f95C4574876d7 client1 6724
0xE295e5D17dbae5FD80b4aBF64678aeA216a9D06d client2 4750
0x67e408fCD53738c39a94220e911204f3c90A787D client3 8612
0x4685B1D9c63717d7060ad689AF77Ba98dFc895E9 client4 4825
0x36D38F3533690B71e1Cc60C9dAbd3A3543813596 client5 7229


In [2]:
from sklearn.preprocessing import MinMaxScaler
import math
import numpy as np
amount = 500
f1scoreDeltaLOO=[-0.08480286336060372, -0.0469214110889411, -0.06094931193456271, 0.3777267063826664, -0.01931377634944098]
scaler=MinMaxScaler()
#Kiểm tra xem số lượng delta âm có nhỏ hơn 5/2 không
deltaPositive = len([i for i in f1scoreDeltaLOO if i > 0])
print(f"deltaPositive: {deltaPositive}")
#Minmax scaling
#f1scoreDeltaLOO=scaler.fit_transform(np.array(f1scoreDeltaLOO).reshape(-1,1)).reshape(-1)
f1scoreDeltaLOO=scaler.fit_transform(np.array(f1scoreDeltaLOO).reshape(-1,1)).reshape(-1)
f1scoreDeltaLOO.sort()
print(f"f1scoreDeltaLOO: {f1scoreDeltaLOO}")
if len([s for s in f1scoreDeltaLOO if s < 0]) >= 5/2:
    # Trung vị
    #f1scoreDeltaLOO.sort()
    median = np.median(f1scoreDeltaLOO)
    for i in range(5):
        # Nếu f1score của client i > median thì lấy tuyệt đối (để chút tính tiền chứ hiện tại đang âm)
        # Ngược lại thì gán bằng 0 -> ko đc trả payoff
        f1scoreDeltaLOO[i] = abs(f1scoreDeltaLOO[i]) if f1scoreDeltaLOO[i] > median else 0
    print(f"f1scoreDeltaLOO: {f1scoreDeltaLOO}")

sumF1scoreDelta = sum(f1scoreDeltaLOO)
print(f"sumF1scoreDelta: {sumF1scoreDelta}")
# Proof of performance (trả theo đóng góp - dựa trên delta)
reward_scores_temp =   [(element / sumF1scoreDelta) * ( amount * 0.8 /5)  for element in f1scoreDeltaLOO]
print(f"reward_scores_temp: {reward_scores_temp}")

# Proof of work (đồng đều, ai cũng được tiền)
reward_scores =  np.full(
  shape=10,
  fill_value=amount * 0.2 / 25,
  dtype=np.float32
)
print(f"reward_scores: {reward_scores}")

# Payoff of each client
payoffByClient = [int(math.floor(x + y)) for x, y in zip(reward_scores_temp, reward_scores)]
print(f"sumPayoff: {payoffByClient}")
sumPayoff = sum(payoffByClient)
print(f"sumPayoff: {sumPayoff}")

deltaPositive: 1
f1scoreDeltaLOO: [0.         0.05157195 0.08190061 0.14158897 1.        ]
sumF1scoreDelta: 1.2750615291028482
reward_scores_temp: [0.0, 3.235730851793762, 5.1386136025161, 8.883585320820945, 62.742070224869195]
reward_scores: [4. 4. 4. 4. 4. 4. 4. 4. 4. 4.]
sumPayoff: [4, 7, 9, 12, 66]
sumPayoff: 98


In [14]:
from sklearn.preprocessing import MinMaxScaler
import math
import numpy as np
amount = 500
id=[2,1]
value=[-0.08480286336060372, -0.0469214110889411]
f1scoreDeltaLOO={id[i]:value[i] for i in range(len(id))}
numClient = 2
rounds = 5
#sort f1scoreDeltaLOO theo value
f1scoreDeltaLOO = dict(sorted(f1scoreDeltaLOO.items(), key=lambda item: item[1]))
#if len([s for s in f1scoreDeltaLOO if s < 0]) >= 5/2:
if len([s for s in f1scoreDeltaLOO.values() if s < 0]) >= numClient/2:
    # Trung vị
    median = np.median(list(f1scoreDeltaLOO.values()))
    for i in f1scoreDeltaLOO:
        # Nếu f1score của client i > median thì lấy tuyệt đối (để chút tính tiền chứ hiện tại đang âm)
        # Ngược lại thì gán bằng 0 -> ko đc trả payoff
        f1scoreDeltaLOO[i] = abs(f1scoreDeltaLOO[i]) if f1scoreDeltaLOO[i] > median else 0
    print(f"f1scoreDeltaLOO: {f1scoreDeltaLOO}")

#sort f1scoreDeltaLOO theo key
f1scoreDeltaLOO = dict(sorted(f1scoreDeltaLOO.items(), key=lambda item: item[0]))
sumF1scoreDelta = sum(f1scoreDeltaLOO.values())
print(f"sumF1scoreDelta: {sumF1scoreDelta}")
# Proof of performance (trả theo đóng góp - dựa trên delta)
reward_scores_temp =   [(element / sumF1scoreDelta) * ( amount * 0.8 /rounds)  for element in f1scoreDeltaLOO.values()]
print(f"reward_scores_temp: {reward_scores_temp}")

# Proof of work (đồng đều, ai cũng được tiền)
reward_scores =  np.full(
  shape=numClient,
  fill_value=amount * 0.2 / rounds*numClient,
  dtype=np.float32
)
print(f"reward_scores: {reward_scores}")

# Payoff of each client
payoffByClient = [int(math.floor(x + y)) for x, y in zip(reward_scores_temp, reward_scores)]
print(f"sumPayoff: {payoffByClient}")
sumPayoff = sum(payoffByClient)
print(f"sumPayoff: {sumPayoff}")

f1scoreDeltaLOO: {2: 0, 1: 0.0469214110889411}
sumF1scoreDelta: 0.0469214110889411
reward_scores_temp: [80.0, 0.0]
reward_scores: [40. 40.]
sumPayoff: [120, 40]
sumPayoff: 160


In [1]:
from sklearn.preprocessing import MinMaxScaler
import math
import numpy as np
amount = 500
id=[2,1,3,5,4]
value=[-0.08480286336060372, -0.0469214110889411, -0.06094931193456271, 0.3777267063826664, -0.01931377634944098]
f1scoreDeltaLOO={id[i]:value[i] for i in range(len(id))}
numClient = 5
rounds = 2
#sort f1scoreDeltaLOO theo value
f1scoreDeltaLOO = dict(sorted(f1scoreDeltaLOO.items(), key=lambda item: item[1]))
print(f"f1scoreDeltaLOO before: {f1scoreDeltaLOO}")
#Minmax scaling các value trong f1scoreDeltaLOO
scaler=MinMaxScaler()
arr=scaler.fit_transform(np.array(list(f1scoreDeltaLOO.values())).reshape(-1,1)).reshape(-1)
#Gán lại value cho f1scoreDeltaLOO
f1scoreDeltaLOO={list(f1scoreDeltaLOO.keys())[i]:arr[i] for i in range(len(arr))}
print(f"f1scoreDeltaLOO after: {f1scoreDeltaLOO}")
if len([s for s in f1scoreDeltaLOO.values() if s < 0]) >= numClient/2:
    # Trung vị
    median = np.median(list(f1scoreDeltaLOO))
    for i in f1scoreDeltaLOO:
        # Nếu f1score của client i > median thì lấy tuyệt đối (để chút tính tiền chứ hiện tại đang âm)
        # Ngược lại thì gán bằng 0 -> ko đc trả payoff
        f1scoreDeltaLOO[i] = abs(f1scoreDeltaLOO[i]) if f1scoreDeltaLOO[i] > median else 0
    print(f"f1scoreDeltaLOO: {f1scoreDeltaLOO}")

#sort f1scoreDeltaLOO theo key
f1scoreDeltaLOO = dict(sorted(f1scoreDeltaLOO.items(), key=lambda item: item[0]))
sumF1scoreDelta = sum(f1scoreDeltaLOO)
print(f"sumF1scoreDelta: {sumF1scoreDelta}")
# Proof of performance (trả theo đóng góp - dựa trên delta)
reward_scores_temp =   [(element / sumF1scoreDelta) * ( amount * 0.8 /rounds)  for element in f1scoreDeltaLOO]
print(f"reward_scores_temp: {reward_scores_temp}")

# Proof of work (đồng đều, ai cũng được tiền)
reward_scores =  np.full(
  shape=numClient,
  fill_value=amount * 0.2 / (numClient*rounds),
  dtype=np.float32
)
print(f"reward_scores: {reward_scores}")

# Payoff of each client
payoffByClient = [int(math.floor(x + y)) for x, y in zip(reward_scores_temp, reward_scores)]
print(f"sumPayoff: {payoffByClient}")
sumPayoff = sum(payoffByClient)
print(f"sumPayoff: {sumPayoff}")

f1scoreDeltaLOO before: {2: -0.08480286336060372, 3: -0.06094931193456271, 1: -0.0469214110889411, 4: -0.01931377634944098, 5: 0.3777267063826664}
f1scoreDeltaLOO after: {2: 0.0, 3: 0.0515719490956677, 1: 0.08190060646866092, 4: 0.14158897353851962, 5: 1.0}
sumF1scoreDelta: 15
reward_scores_temp: [13.333333333333334, 26.666666666666668, 40.0, 53.333333333333336, 66.66666666666666]
reward_scores: [10. 10. 10. 10. 10.]
sumPayoff: [23, 36, 50, 63, 76]
sumPayoff: 248


In [1]:
arr={'a':1,'b':2}
temp={'a':3,'b':3}
#cộng dict temp vào arr, nếu key đã tồn tại thì cộng giá trị
for key in temp:
    if key in arr:
        arr[key]+=temp[key]
    else:
        arr[key]=temp[key]
print(arr)

{'a': 4, 'b': 5}
